# Check `backend/RAG.py` Output

This notebook runs one end-to-end RAG query against the existing Chroma collection and prints the answer, document sources, graph sources, token usage, and optional graph error.

Before running it, make sure `OPENAI_API_KEY` is available in `/workspace/.env` or in your shell environment. If Neo4j is not running, `RAG.py` will still answer from document retrieval and place the graph issue in `graph_error`.

In [7]:
from pathlib import Path
import json
import os
import sys

from dotenv import load_dotenv


In [8]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=False)

if not (os.getenv("OPENAI_API_KEY") or "").strip():
    raise ValueError("OPENAI_API_KEY is missing. Add it to /workspace/.env or your environment.")

print(f"Project root: {PROJECT_ROOT}")


Project root: /workspace


In [15]:
from backend.RAG import rag

query = "In the sale deed, who sold the property at Kerkstraat 12 and what was the sale price?"

result = rag(
    query=query,
    top_k=3,
)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [16]:
print("QUESTION")
print(query)

print("\nANSWER")
print(result["answer"])

print("\nDOCUMENT SOURCES")
print(json.dumps(result["sources"], indent=2, ensure_ascii=False))

print("\nGRAPH SOURCES")
print(json.dumps(result["graph_sources"], indent=2, ensure_ascii=False))

if result.get("graph_error"):
    print("\nGRAPH ERROR")
    print(result["graph_error"])

print("\nRUN INFO")
print(json.dumps({
    "model_used": result["model_used"],
    "embedding_model": result["embedding_model"],
    "reranker_model": result["reranker_model"],
    "collection": result["collection"],
    "top_k": result["top_k"],
    "response_time_seconds": result["response_time_seconds"],
    "prompt_tokens": result["prompt_tokens"],
    "completion_tokens": result["completion_tokens"],
    "total_tokens": result["total_tokens"],
}, indent=2, ensure_ascii=False))


QUESTION
In the sale deed, who sold the property at Kerkstraat 12 and what was the sale price?

ANSWER
The property at Kerkstraat 12 was sold by the Sellers for a sale price of EUR 320,000 [Document 1][Document 2]. However, the specific names of the Sellers are not provided in the context given. Therefore, it is unclear who exactly sold the property.

DOCUMENT SOURCES
[
  {
    "rank": 1,
    "chunk_id": "sale_004-p001-c0001",
    "document_id": "sale_004",
    "document_type": "sale_deed",
    "page_number": 1,
    "distance": 0.8195586086260427,
    "rerank_score": 5.782299041748047
  },
  {
    "rank": 2,
    "chunk_id": "sale_004-p002-c0002",
    "document_id": "sale_004",
    "document_type": "sale_deed",
    "page_number": 2,
    "distance": 0.664478592661318,
    "rerank_score": 5.381174564361572
  },
  {
    "rank": 3,
    "chunk_id": "purchase_001-p001-c0001",
    "document_id": "purchase_001",
    "document_type": "purchase_contract",
    "page_number": 1,
    "distance": 0.8

To inspect the exact prompt sent to the chat model, run the optional cell below.

In [6]:
print(result["prompt"])


You are an estate-document assistant.
Answer the QUESTION based only on the CONTEXT from retrieved notarial documents and graph facts.
If the context is insufficient, clearly say what is missing and do not invent details.
When you cite facts, mention source ids in square brackets like [Document 1] or [Graph 1].

QUESTION:
In the sale deed, who sold the property at Kerkstraat 12 and what was the sale price?

CONTEXT:
DOCUMENT RETRIEVAL:
[Document 1]
chunk_id: sale_004-p001-c0001
document_id: sale_004
document_type: sale_deed
page_number: 1
distance: 0.819559
rerank_score: 5.782299
people: []
dates: ['12\nSeptember 1985']
years: [1985]
amounts_eur: ['320,000']
roles: ['buyer', 'notary', 'seller']
text:
hereinafter referred to as the "Buyers".

---

**RECITALS:**

WHEREAS, the Sellers are the lawful owners of the property located at Kerkstraat 12, 8000 Bruges,
Belgium, hereinafter referred to as the "Property", acquired by them through a purchase deed dated 12
September 1985, which is rec